#### Loading dataset and filtering it

In [84]:
import pandas as pd
from bertopic import BERTopic

In [85]:
#Reading csv 
df = pd.read_csv(r'dataset\datasetA.csv')
#Checking first 5 rows
df.head()

,title,link,date,source,number_of_characters_title,number_of_words_title,day_of_week,month,year,quarter,is_weekend,classes_str
0,Google’s AI is the ‘worst’ for stealing conten...,https://news.google.com/rss/articles/CBMipgFBV...,2025-09-11,Fortune,74,13,Thursday,September,2025,3,False,Sentiment (Positive / Negative Feelings); Huma...
1,Powering the Next Wave of Enterprise Innovatio...,https://news.google.com/rss/articles/CBMitgFBV...,2025-09-11,Silicon Canals,106,16,Thursday,September,2025,3,False,"Creativity, Expression & Identity; Work, Jobs ..."
2,AI a ‘strategic necessity’ law lecturer says,https://news.google.com/rss/articles/CBMiiAFBV...,2025-09-11,qlsproctor.com.au,64,9,Thursday,September,2025,3,False,"Society, Ethics & Culture"
3,Datacom sees AI agents as pivotal to legacy ap...,https://news.google.com/rss/articles/CBMirAFBV...,2025-09-11,ARNnet,70,12,Thursday,September,2025,3,False,"Routine, Lifestyle & Behavior"
4,"Student Blog: Startups, AI, and Lessons from S...",https://news.google.com/rss/articles/CBMijwFBV...,2025-09-11,The University of Queensland,85,13,Thursday,September,2025,3,False,"Learning, Knowledge & Education"


In [86]:
#Selecting just the title and classes_str columns. 
keyData = df[['title', 'classes_str']]
#Checking nulls 
keyData.isna().sum()

title          0
classes_str    0
dtype: int64

In [87]:
#Filtering the df so it only selects rows where the classes_str value contains 'Learning, Knowledge & Education'
df2 = keyData[keyData['classes_str'].str.contains('Learning, Knowledge & Education', case=False)]
df2.head()

,title,classes_str
4,"Student Blog: Startups, AI, and Lessons from S...","Learning, Knowledge & Education"
16,West Alabama school district looks to strength...,"Human Roles; Learning, Knowledge & Education; ..."
24,CEO: Proposed data center in College Station f...,"Learning, Knowledge & Education"
25,Maine Monitor: ‘Building the plane as we’re fl...,"Learning, Knowledge & Education"
42,Singtel to accelerate employees' AI upskilling...,"Work, Jobs & Economy; Learning, Knowledge & Ed..."


#### Topic Modeling 

In [88]:
#BERTopic only reads lists so i did this
docs = df2["title"].tolist()
print(len(docs))

1946


In [89]:
#pre making embeddings
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedding_model.encode(docs)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 419.19it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [93]:
import numpy as np
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

#makes everything determinisitc
np.random.seed(42)

umap_model = UMAP(n_neighbors=5, n_components=5, min_dist=0.0, metric="cosine", random_state=42)

hdbscan_model = HDBSCAN(
    min_cluster_size=20,   # increase for fewer topics
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

representation_model = KeyBERTInspired()
topic_model = BERTopic(embedding_model=embedding_model, umap_model=umap_model, hdbscan_model=hdbscan_model, 
                       representation_model=representation_model, verbose=True)
topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

topic_model.reduce_topics(docs, nr_topics=15)
tm = topic_model.get_topic_info()
tm


2026-01-30 17:58:23,294 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-01-30 17:58:34,043 - BERTopic - Dimensionality - Completed ✓
2026-01-30 17:58:34,045 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-01-30 17:58:34,169 - BERTopic - Cluster - Completed ✓
2026-01-30 17:58:34,174 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-30 17:58:35,617 - BERTopic - Representation - Completed ✓
2026-01-30 17:58:35,711 - BERTopic - Topic reduction - Reducing number of topics
2026-01-30 17:58:35,712 - BERTopic - Topic reduction - Number of topics (15) is equal or higher than the clustered topics(15).
2026-01-30 17:58:35,713 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-01-30 17:58:37,524 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,448,-1_ai_learning_education_intelligence,"[ai, learning, education, intelligence, curric...",[Navigating lifelong learning in the age of ar...
1,0,315,0_ai_education_curriculum_intelligence,"[ai, education, curriculum, intelligence, acad...",[The Complete Guide to Using AI in the Educati...
2,1,281,1_classrooms_classroom_ai_educators,"[classrooms, classroom, ai, educators, educati...",[The future of artificial intelligence in educ...
3,2,221,2_ai_education_openai_universities,"[ai, education, openai, universities, google, ...","[Teachers union partners with Anthropic, Micro..."
4,3,183,3_ai_medical_physicians_intelligence,"[ai, medical, physicians, intelligence, doctor...",[Application and ethical implication of genera...
5,4,98,4_chatbots_chatbot_ai_chatgpt,"[chatbots, chatbot, ai, chatgpt, chat, convers...",[Study says AI chatbots need to fix suicide re...
6,5,92,5_creativity_ai_generative_creative,"[creativity, ai, generative, creative, intelli...",[How generative AI is transforming beauty: the...
7,6,69,6_lawsuit_sued_sue_copyright,"[lawsuit, sued, sue, copyright, ai, law, copyr...",[Federal judge sides with Meta in lawsuit over...
8,7,51,7_ai_jobs_hiring_workplace,"[ai, jobs, hiring, workplace, stanford, tech, ...",[These Fields Are Losing the Most Entry-Level ...
9,8,47,8_ai_education_intelligence_learning,"[ai, education, intelligence, learning, teache...","[Watch: AI will aid with NCEA replacement, Edu..."


In [94]:
with open("Topics.txt", 'w') as f:
    for i in range(1, 6):
        f.write(f"Topic Name: {tm['Name'].iloc[i]} \n")
        f.write(f"Document Count For Topic: {tm['Count'].iloc[i]} \n")
        f.write(f"Representation and Weights: \n")
        for tup in topic_model.get_topic(i):
            f.write(f"({tup[0]}, {float(tup[1]):.2f}) \n")
        f.write("Representative Docs: \n")
        for doc in tm['Representative_Docs'].iloc[i]:
            f.write(f"{doc} \n")
        f.write("\n")

#### Sentiment Analysis

In [95]:
docDf = topic_model.get_document_info(docs)
docDf.head()

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,"Student Blog: Startups, AI, and Lessons from S...",0,0_ai_education_curriculum_intelligence,"[ai, education, curriculum, intelligence, acad...",[The Complete Guide to Using AI in the Educati...,ai - education - curriculum - intelligence - a...,1.000000,False
1,West Alabama school district looks to strength...,1,1_classrooms_classroom_ai_educators,"[classrooms, classroom, ai, educators, educati...",[The future of artificial intelligence in educ...,classrooms - classroom - ai - educators - educ...,0.637837,False
2,CEO: Proposed data center in College Station f...,0,0_ai_education_curriculum_intelligence,"[ai, education, curriculum, intelligence, acad...",[The Complete Guide to Using AI in the Educati...,ai - education - curriculum - intelligence - a...,1.000000,False
3,Maine Monitor: ‘Building the plane as we’re fl...,1,1_classrooms_classroom_ai_educators,"[classrooms, classroom, ai, educators, educati...",[The future of artificial intelligence in educ...,classrooms - classroom - ai - educators - educ...,0.910565,False
4,Singtel to accelerate employees' AI upskilling...,2,2_ai_education_openai_universities,"[ai, education, openai, universities, google, ...","[Teachers union partners with Anthropic, Micro...",ai - education - openai - universities - googl...,1.000000,False


##### Vader Sentiment Analysis (Rule Based)

In [96]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
def sentiment_scores(text):
    sid_obj = SentimentIntensityAnalyzer()
    sentiment_dict = sid_obj.polarity_scores(text)
    return sentiment_dict

def getSentiment(compound):
    if compound >= 0.05:
        return "Positive"
    elif compound <= -0.05:
        return "Negative"
    else:
        return "Neutral"

In [97]:
topic1 = docDf[docDf['Topic'] == 0]
v_sent_count = {
    "Negative": 0,
    "Neutral": 0,
    "Positive": 0
}

with open("Vader Sentiment Analysis.txt", "w", encoding="utf-8", errors="replace") as f:
    for doc in topic1['Document']:
        f.write(f"Article Title: {doc} \n")
        scores = sentiment_scores(doc)
        f.write(f"Sentiment: {getSentiment(scores['compound'])} \n")
        v_sent_count[getSentiment(scores['compound'])] += 1
        f.write("\n")

v_sent_count

{'Negative': 26, 'Neutral': 136, 'Positive': 153}

##### Text Blob Sentiment Analysis (Machine Learning Based)

In [ ]:
from textblob import TextBlob
def get_sentiment_label (polarity):
    if polarity > 0.2:
        return 'Positive'
    elif polarity < -0.2:
        return 'Negative'
    else:
        return 'Neutral'

In [ ]:
tb_sent_count = {
    "Negative": 0,
    "Neutral": 0,
    "Positive": 0
}

with open("Textblob Sentiment Analysis.txt", "w", encoding="utf-8", errors="replace") as f:
    for doc in topic1['Document']:
        f.write(f"Article Title: {doc} \n")
        docBlob = TextBlob(doc)
        polarity = docBlob.sentiment[0]
        sent = get_sentiment_label(polarity)
        f.write(f"Polarity: {polarity} \n")
        f.write(f"Sentiment: {sent} \n")
        tb_sent_count[sent] += 1
        f.write("\n")

tb_sent_count

{'Negative': 64, 'Neutral': 220, 'Positive': 38}